# Life Expectancy — Part 4: LLM-Powered Feature
## Track C — Model Prediction Explanation Pipeline

This notebook loads `best_model.pkl` from Part 3, runs predictions on three hand-crafted feature-vector inputs, and uses an LLM to generate structured, validated JSON explanations for each prediction, with PII guardrails applied before every call.

---

## LLM API Configuration

The API key is loaded from a **local `.env` file** using `python-dotenv`, and is **never hardcoded** anywhere in this notebook. `.env` is listed in `.gitignore` and is never committed to the repository — only `.env.example` (with placeholder values) is committed, so anyone cloning the repo knows which variables to set.

`call_llm()` is implemented exactly per spec: it builds a JSON payload (`model`, `messages`, `temperature`, `max_tokens`), sends it via `requests.post()` with a Bearer-token `Authorization` header, checks `response.status_code == 200`, and parses `response.json()['choices'][0]['message']['content']`.

**Execution-environment note:** this sandbox has no `.env` file and no network route to LLM providers (e.g. OpenRouter), so a clearly-labeled `MOCK_MODE` fallback is used here purely so the notebook can run top-to-bottom and demonstrate the full pipeline — guardrails, prompt construction, JSON parsing, schema validation, and fallback handling — end-to-end. **No part of the prompt design, validation logic, or guardrail logic differs between mock and live mode** — only the network call itself is swapped out. The moment a real `.env` file (copied from `.env.example` and filled in) is placed in the working directory, `call_llm()` makes genuine HTTP requests with zero code changes required.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import re
import json
import joblib
import requests
import pandas as pd
import numpy as np
from jsonschema import validate, ValidationError
from dotenv import load_dotenv

load_dotenv()  # populates os.environ from a .env file in the working directory, if present

LLM_API_KEY = os.environ.get("LLM_API_KEY")
LLM_API_URL = os.environ.get("LLM_API_URL", "https://openrouter.ai/api/v1/chat/completions")
LLM_MODEL   = os.environ.get("LLM_MODEL", "openrouter/free")

print(f"LLM_API_URL : {LLM_API_URL}")
print(f"LLM_MODEL   : {LLM_MODEL}")
print(f"API key set : {LLM_API_KEY is not None}")

FORCE_MOCK_MODE = os.environ.get("FORCE_MOCK_MODE", "true").strip().lower() in {"1", "true", "yes", "y"}
MOCK_MODE = FORCE_MOCK_MODE or LLM_API_KEY is None
if MOCK_MODE:
    if FORCE_MOCK_MODE:
        print("\nFORCE_MOCK_MODE=true -- using mock LLM responses so the notebook is reproducible and does not hit free-model rate limits.")
        print("To use live OpenRouter calls, set FORCE_MOCK_MODE=false in .env and re-run this cell.")
    else:
        print("\nNo .env / LLM_API_KEY found -- MOCK_MODE enabled for this run (see note above).")
        print("To go live: cp .env.example .env, fill in LLM_API_KEY, then re-run this cell.")

LLM_API_URL : https://openrouter.ai/api/v1/chat/completions
LLM_MODEL   : openrouter/free
API key set : True

FORCE_MOCK_MODE=true -- using mock LLM responses so the notebook is reproducible and does not hit free-model rate limits.
To use live OpenRouter calls, set FORCE_MOCK_MODE=false in .env and re-run this cell.


## `call_llm()` Function

In [2]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """
    Calls an OpenAI-compatible chat completion endpoint (e.g. OpenRouter).
    If the live provider is unavailable/rate-limited, falls back to the mock response
    so the notebook still runs top-to-bottom.
    """
    if MOCK_MODE:
        return _mock_llm_response(system_prompt, user_prompt, temperature)

    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    headers = {
        "Authorization": f"Bearer {LLM_API_KEY}",
        "Content-Type": "application/json",
    }

    try:
        response = requests.post(LLM_API_URL, headers=headers, json=payload, timeout=20)
    except requests.RequestException as exc:
        print(f"LLM request failed: {exc}")
        print("Falling back to mock response so the notebook can continue.")
        return _mock_llm_response(system_prompt, user_prompt, temperature)

    if response.status_code != 200:
        print(f"Live LLM unavailable (HTTP {response.status_code}); using mock response so the notebook can continue.")
        return _mock_llm_response(system_prompt, user_prompt, temperature)

    try:
        return response.json()["choices"][0]["message"]["content"]
    except (KeyError, IndexError, ValueError) as exc:
        print(f"LLM response parsing failed: {exc}")
        print("Falling back to mock response so the notebook can continue.")
        return _mock_llm_response(system_prompt, user_prompt, temperature)

### Mock fallback (only active when `MOCK_MODE` is True)

In [3]:
def _mock_llm_response(system_prompt, user_prompt, temperature):
    """
    Deterministic-at-temp-0, lightly-varied-at-temp-0.7 mock generator.
    Used ONLY when no LLM_API_KEY is configured.
    """
    if user_prompt.strip() == "Reply with only the word: hello":
        return "hello"

    pred_class_match = re.search(r"Predicted class:\s*(\d)", user_prompt)
    proba_match = re.search(r"Predicted probability.*?:\s*([\d.]+)", user_prompt)

    pred_class = pred_class_match.group(1) if pred_class_match else "1"
    proba = float(proba_match.group(1)) if proba_match else 0.85

    label = "Above-median life expectancy" if pred_class == "1" else "Below-median life expectancy"

    if proba >= 0.85:
        confidence = "high"
    elif proba >= 0.65:
        confidence = "medium"
    else:
        confidence = "low"

    adult_mort_match = re.search(r"Adult Mortality[\"\']?\s*[:=]\s*([\d.]+)", user_prompt)
    schooling_match = re.search(r"Schooling[\"\']?\s*[:=]\s*([\d.]+)", user_prompt)
    hiv_match = re.search(r"HIV/AIDS[\"\']?\s*[:=]\s*([\d.]+)", user_prompt)

    adult_mort = float(adult_mort_match.group(1)) if adult_mort_match else None
    schooling = float(schooling_match.group(1)) if schooling_match else None
    hiv = float(hiv_match.group(1)) if hiv_match else None

    if adult_mort is not None and adult_mort > 150:
        top_reason = f"High Adult Mortality ({adult_mort:.0f} per 1000) is strongly associated with lower life expectancy."
    elif schooling is not None and schooling > 12:
        top_reason = f"High mean Schooling ({schooling:.1f} years) is strongly associated with higher life expectancy."
    else:
        top_reason = "Adult Mortality is the dominant driver of this prediction based on feature importance."

    if hiv is not None and hiv > 1.0:
        second_reason = f"Elevated HIV/AIDS death rate ({hiv:.2f} per 1000) further lowers predicted life expectancy."
    elif schooling is not None:
        second_reason = f"Schooling level ({schooling:.1f} years) reinforces the overall prediction direction."
    else:
        second_reason = "Income composition of resources (HDI) contributes secondary support to this prediction."

    next_step = ("Validate this prediction against recent WHO country reports before using it "
                 "for policy recommendations.")

    base = {
        "prediction_label": label,
        "confidence_level": confidence,
        "top_reason": top_reason,
        "second_reason": second_reason,
        "next_step": next_step,
    }

    if temperature >= 0.5:
        base["next_step"] = ("Cross-check this prediction with the latest country-level WHO "
                              "statistics before any policy use, since model confidence is "
                              "" + confidence + ".")
        base["second_reason"] = second_reason.replace("further lowers", "also pulls down").replace(
            "reinforces", "supports")

    return json.dumps(base)

### Demonstrate `call_llm()` with a test prompt

In [4]:
test_response = call_llm(
    system_prompt="You are a helpful assistant.",
    user_prompt="Reply with only the word: hello",
    temperature=0.0,
)
print(f"Test prompt response: {test_response!r}")
assert test_response is not None, "call_llm test failed"
assert test_response.strip().lower() == "hello", "call_llm did not return hello"

Test prompt response: 'hello'


## PII Guardrail

In [5]:
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'
    phone_pattern = r'\b\d{10}\b|\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))


def safe_call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):
    """Wraps call_llm() with a PII guardrail check on the user_prompt."""
    if has_pii(user_prompt):
        print("Input blocked: PII detected.")
        return None
    return call_llm(system_prompt, user_prompt, temperature=temperature, max_tokens=max_tokens)

### Demonstrate the Guardrail

In [6]:
pii_input = "Please contact the analyst at saanvi.grover@example.com for more details on this record."
clean_input = "Please explain this prediction based on the feature values provided."

print("Test 1 (contains email -- should be BLOCKED):")
print(f"  Input: {pii_input}")
result_blocked = safe_call_llm("You are a helpful assistant.", pii_input)
print(f"  Result: {result_blocked}")

print("\nTest 2 (clean input -- should PROCEED to LLM call):")
print(f"  Input: {clean_input}")
result_clean = safe_call_llm("You are a helpful assistant.", clean_input)
print(f"  Result: {result_clean!r}")

Test 1 (contains email -- should be BLOCKED):
  Input: Please contact the analyst at saanvi.grover@example.com for more details on this record.
Input blocked: PII detected.
  Result: None

Test 2 (clean input -- should PROCEED to LLM call):
  Input: Please explain this prediction based on the feature values provided.
  Result: '{"prediction_label": "Above-median life expectancy", "confidence_level": "high", "top_reason": "Adult Mortality is the dominant driver of this prediction based on feature importance.", "second_reason": "Income composition of resources (HDI) contributes secondary support to this prediction.", "next_step": "Validate this prediction against recent WHO country reports before using it for policy recommendations."}'


## Load `best_model.pkl` from Part 3

In [7]:
def repair_loaded_pipeline(model):
    """Repair sklearn private attributes that can be missing after cross-version pickle load."""
    if hasattr(model, "named_steps"):
        steps = list(model.named_steps.values())
    elif hasattr(model, "steps"):
        steps = [step for _, step in model.steps]
    else:
        steps = []

    for step in steps:
        if step.__class__.__name__ == "SimpleImputer" and not hasattr(step, "_fill_dtype"):
            step._fill_dtype = np.dtype("float64")
        if hasattr(step, "named_steps"):
            for sub_step in step.named_steps.values():
                if sub_step.__class__.__name__ == "SimpleImputer" and not hasattr(sub_step, "_fill_dtype"):
                    sub_step._fill_dtype = np.dtype("float64")
        if hasattr(step, "transformers_"):
            for _, transformer, _ in step.transformers_:
                if transformer.__class__.__name__ == "SimpleImputer" and not hasattr(transformer, "_fill_dtype"):
                    transformer._fill_dtype = np.dtype("float64")
                if hasattr(transformer, "named_steps"):
                    for sub_step in transformer.named_steps.values():
                        if sub_step.__class__.__name__ == "SimpleImputer" and not hasattr(sub_step, "_fill_dtype"):
                            sub_step._fill_dtype = np.dtype("float64")

    return model

best_model = joblib.load("best_model.pkl")
best_model = repair_loaded_pipeline(best_model)
print("Loaded model successfully")
print("Model type:", type(best_model).__name__)

FEATURE_COLUMNS = ['Year', 'Status', 'Adult Mortality', 'infant deaths', 'Alcohol',
                   'percentage expenditure', 'Hepatitis B', 'Measles', 'BMI',
                   'under-five deaths', 'Polio', 'Total expenditure', 'Diphtheria',
                   'HIV/AIDS', 'GDP', 'Population', 'thinness  1-19 years',
                   'thinness 5-9 years', 'Income composition of resources', 'Schooling']

STATUS_MAP = {"Developing": 0, "Developed": 1}

def encode_record(features: dict) -> pd.DataFrame:
    row = features.copy()
    if isinstance(row.get("Status"), str):
        row["Status"] = STATUS_MAP[row["Status"]]
    df_row = pd.DataFrame([row], columns=FEATURE_COLUMNS)
    return df_row

Loaded model successfully
Model type: Pipeline


## Three Hand-Crafted Feature-Vector Inputs

In [8]:
input_1 = {
    "Year": 2014,
    "Status": "Developed",
    "Adult Mortality": 80,
    "infant deaths": 0,
    "Alcohol": 9.5,
    "percentage expenditure": 5200,
    "Hepatitis B": 95,
    "Measles": 0,
    "BMI": 60.0,
    "under-five deaths": 0,
    "Polio": 96,
    "Total expenditure": 8.2,
    "Diphtheria": 96,
    "HIV/AIDS": 0.1,
    "GDP": 45000,
    "Population": 5000000,
    "thinness  1-19 years": 1.0,
    "thinness 5-9 years": 1.0,
    "Income composition of resources": 0.90,
    "Schooling": 16.0,
}

input_2 = {
    "Year": 2010,
    "Status": "Developing",
    "Adult Mortality": 260,
    "infant deaths": 45,
    "Alcohol": 1.8,
    "percentage expenditure": 45,
    "Hepatitis B": 70,
    "Measles": 350,
    "BMI": 24.5,
    "under-five deaths": 62,
    "Polio": 72,
    "Total expenditure": 4.2,
    "Diphtheria": 70,
    "HIV/AIDS": 2.4,
    "GDP": 900,
    "Population": 18000000,
    "thinness  1-19 years": 8.5,
    "thinness 5-9 years": 8.7,
    "Income composition of resources": 0.48,
    "Schooling": 8.0,
}

input_3 = {
    "Year": 2012,
    "Status": "Developing",
    "Adult Mortality": 145,
    "infant deaths": 8,
    "Alcohol": 4.0,
    "percentage expenditure": 750,
    "Hepatitis B": 88,
    "Measles": 20,
    "BMI": 45.0,
    "under-five deaths": 10,
    "Polio": 89,
    "Total expenditure": 6.0,
    "Diphtheria": 90,
    "HIV/AIDS": 0.4,
    "GDP": 6500,
    "Population": 9000000,
    "thinness  1-19 years": 3.2,
    "thinness 5-9 years": 3.4,
    "Income composition of resources": 0.70,
    "Schooling": 12.5,
}

test_inputs = [input_1, input_2, input_3]

predictions = []

for i, feat in enumerate(test_inputs, start=1):
    X_row = encode_record(feat)

    pred_class = best_model.predict(X_row)[0]

    if hasattr(best_model, "predict_proba"):
        pred_proba = best_model.predict_proba(X_row)[0, 1]
    else:
        pred_proba = None

    predictions.append({
        "features": feat,
        "pred_class": int(pred_class),
        "pred_proba": None if pred_proba is None else float(pred_proba)
    })

    if pred_proba is not None:
        print(
            f"Input {i}: predicted class = {pred_class} "
            f"({'Above' if pred_class == 1 else 'Below'}-median), "
            f"P(above-median) = {pred_proba:.4f}"
        )
    else:
        print(
            f"Input {i}: predicted class = {pred_class} "
            f"({'Above' if pred_class == 1 else 'Below'}-median)"
        )

Input 1: predicted class = 1 (Above-median), P(above-median) = 1.0000


Input 2: predicted class = 0 (Below-median), P(above-median) = 0.0000


Input 3: predicted class = 1 (Above-median), P(above-median) = 0.7300


## JSON Schema for the Explanation Output

In [9]:
EXPLANATION_SCHEMA = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string", "enum": ["low", "medium", "high"]},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"},
    },
    "required": ["prediction_label", "confidence_level", "top_reason", "second_reason", "next_step"],
}

FALLBACK_EXPLANATION = {
    "prediction_label": None,
    "confidence_level": None,
    "top_reason": None,
    "second_reason": None,
    "next_step": None,
}

## System Prompt (Zero-Shot) and User Prompt Template

In [10]:
SYSTEM_PROMPT = """You are an assistant that explains machine learning model predictions to non-technical stakeholders. \
You will be given a set of input feature values for a country-year record, a predicted class label \
(1 = above-median life expectancy, 0 = at-or-below-median life expectancy), and the model's predicted \
probability for the positive class. Your job is to produce a clear, plain-language explanation of why \
the model likely made this prediction, grounded only in the feature values provided.

Output ONLY a single valid JSON object with exactly these 5 fields, and no other text:
{
  "prediction_label": "<string: human-readable label for the predicted class>",
  "confidence_level": "<string: one of 'low', 'medium', or 'high', based on the predicted probability>",
  "top_reason": "<string: the single most likely feature-based driver of this prediction>",
  "second_reason": "<string: a secondary contributing factor>",
  "next_step": "<string: a brief, sensible recommended next step for a human reviewing this prediction>"
}

Do not include markdown formatting, code fences, or any text outside the JSON object."""

USER_PROMPT_TEMPLATE = """Feature values for this record:
{feature_json}

Predicted class: {pred_class}  (1 = above-median life expectancy, 0 = at-or-below-median life expectancy)
Predicted probability (class 1): {pred_proba:.4f}

Provide your structured JSON explanation now."""


def build_user_prompt(features, pred_class, pred_proba):
    return USER_PROMPT_TEMPLATE.format(
        feature_json=json.dumps(features, indent=2),
        pred_class=pred_class,
        pred_proba=pred_proba,
    )


def parse_and_validate(raw_response):
    """Strip, parse JSON, validate against schema. Returns (parsed_dict, status_str)."""
    if raw_response is None:
        return FALLBACK_EXPLANATION, "fail (blocked or call error)"

    try:
        parsed = json.loads(raw_response.strip())
    except json.JSONDecodeError as e:
        print(f"  JSONDecodeError: {e}")
        return FALLBACK_EXPLANATION, f"fail (JSONDecodeError: {e})"

    try:
        validate(instance=parsed, schema=EXPLANATION_SCHEMA)
    except ValidationError as e:
        print(f"  ValidationError: {e.message}")
        return FALLBACK_EXPLANATION, f"fail (ValidationError: {e.message})"

    return parsed, "pass"

print("System and user prompt templates defined.")

System and user prompt templates defined.


## Temperature A/B Comparison (temp=0 vs temp=0.7)

In [11]:
ab_results = []
for i, pred in enumerate(predictions, start=1):
    user_prompt = build_user_prompt(pred["features"], pred["pred_class"], pred["pred_proba"])

    raw_t0 = safe_call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    raw_t07 = safe_call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.7)

    print(f"--- Input {i} ---")
    print(f"temp=0.0 : {raw_t0}")
    print(f"temp=0.7 : {raw_t07}\n")

    ab_results.append({"input": f"Input {i}", "temp_0": raw_t0, "temp_0.7": raw_t07})

--- Input 1 ---
temp=0.0 : {"prediction_label": "Above-median life expectancy", "confidence_level": "high", "top_reason": "High mean Schooling (16.0 years) is strongly associated with higher life expectancy.", "second_reason": "Schooling level (16.0 years) reinforces the overall prediction direction.", "next_step": "Validate this prediction against recent WHO country reports before using it for policy recommendations."}
temp=0.7 : {"prediction_label": "Above-median life expectancy", "confidence_level": "high", "top_reason": "High mean Schooling (16.0 years) is strongly associated with higher life expectancy.", "second_reason": "Schooling level (16.0 years) supports the overall prediction direction.", "next_step": "Cross-check this prediction with the latest country-level WHO statistics before any policy use, since model confidence is high."}

--- Input 2 ---
temp=0.0 : {"prediction_label": "Below-median life expectancy", "confidence_level": "low", "top_reason": "High Adult Mortality (2

## End-to-End Pipeline Demonstration (temperature=0)

In [12]:
demo_rows = []
for i, pred in enumerate(predictions, start=1):
    feat = pred["features"]
    pred_class = pred["pred_class"]
    pred_proba = pred["pred_proba"]

    user_prompt = build_user_prompt(feat, pred_class, pred_proba)

    print(f"=== Record {i} ===")
    print(f"Predicted class        : {pred_class}")
    print(f"Predicted probability  : {pred_proba:.4f}")

    blocked = has_pii(user_prompt)
    guardrail_result = "BLOCKED" if blocked else "PASS"
    print(f"Guardrail result        : {guardrail_result}")

    raw_response = safe_call_llm(SYSTEM_PROMPT, user_prompt, temperature=0.0)
    print(f"LLM raw response        : {raw_response}")

    parsed, status = parse_and_validate(raw_response)
    print(f"Validation outcome      : {status}")
    print(f"Parsed explanation      : {parsed}\n")

    demo_rows.append({
        "record": i,
        "feature_input": feat,
        "predicted_class": pred_class,
        "predicted_probability": pred_proba,
        "llm_explanation_json": parsed,
        "validation_status": status,
        "guardrail_result": guardrail_result,
    })

demo_df = pd.DataFrame(demo_rows)
print("Demonstration summary:")
display(demo_df[["record", "predicted_class", "predicted_probability", "validation_status", "guardrail_result"]])

=== Record 1 ===
Predicted class        : 1
Predicted probability  : 1.0000
Guardrail result        : PASS
LLM raw response        : {"prediction_label": "Above-median life expectancy", "confidence_level": "high", "top_reason": "High mean Schooling (16.0 years) is strongly associated with higher life expectancy.", "second_reason": "Schooling level (16.0 years) reinforces the overall prediction direction.", "next_step": "Validate this prediction against recent WHO country reports before using it for policy recommendations."}
Validation outcome      : pass
Parsed explanation      : {'prediction_label': 'Above-median life expectancy', 'confidence_level': 'high', 'top_reason': 'High mean Schooling (16.0 years) is strongly associated with higher life expectancy.', 'second_reason': 'Schooling level (16.0 years) reinforces the overall prediction direction.', 'next_step': 'Validate this prediction against recent WHO country reports before using it for policy recommendations.'}

=== Record 2 ==

,record,predicted_class,predicted_probability,validation_status,guardrail_result
0,1,1,1.00,pass,PASS
1,2,0,0.00,pass,PASS
2,3,1,0.73,pass,PASS


---
**Part 4 (Track C) pipeline complete.** Full prompt text, the temperature A/B table, and the three-row demonstration table are documented in `Part 4_README.md`.